# Query Rewriting RAG
### Reformulating a vague query before it gets embedded

Corpus: `OWASP Top 10 for LLM Applications (2025)` — 10 named risk categories (LLM01–LLM10) sharing vocabulary like “risk”, “attack”, “model”, which is exactly what makes naive retrieval struggle.

## Step 1: Build the pipeline
PDF → Chunks → Embeddings → FAISS (same setup as `Simple_RAG.ipynb`).

In [1]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf -q


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama

C:\Users\shiva\AppData\Local\Temp\ipykernel_7536\1805325906.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


C:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
PDF_PATH = "OWASP-Top-10-for-LLMs-v2025.pdf"

pages = PyPDFLoader(PDF_PATH).load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

# Drop table-of-contents chunks: dotted leaders ("Prevention  .  .  .  .  12") are literal
# text in the PDF, so bare category-name queries (e.g. "Unbounded Consumption") match the
# TOC listing itself just as well as the real section content, unless we filter it out here.
def is_toc_chunk(text, threshold=0.15):
    return (text.count(". ") + text.count(".  ")) / max(len(text), 1) > threshold

chunks = [c for c in chunks if not is_toc_chunk(c.page_content)]

embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vector_store = FAISS.from_documents(chunks, embeddings)

llm = ChatOllama(model="llama3.2:3b", temperature=0)

print(f"Loaded {len(pages)} pages -> {len(chunks)} chunks -> {vector_store.index.ntotal} vectors")

incorrect startxref pointer(1)


parsing for Object Streams


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Loaded 45 pages -> 116 chunks -> 116 vectors


## Step 2: Baseline — embed the vague query as-is
Users rarely phrase questions the way a technical document does. A short, conversational question is a poor match for formal prose.

In [4]:
query = "Our AI keeps leaking private customer data like emails and account numbers in its replies"

print("Baseline (raw vague query):")
for doc in vector_store.similarity_search(query, k=3):
    print(f"page {doc.metadata['page']}: {doc.page_content}...")

Baseline (raw vague query):
page 8: Example Attack Scenarios
Scenario #1: Direct Injection
An attacker injects a prompt into a customer support chatbot, instructing it to ignore
previous guidelines, query private data stores, and send emails, leading to unauthorized
access and privilege escalation.
Scenario #2: Indirect Injection
A user employs an LLM to summarize a webpage containing hidden instructions that cause
the LLM to insert an image linking to a URL, leading to exfiltration of the the private
conversation.
Scenario #3: Unintentional Injection
A company includes an instruction in a job description to identify AI-generated applications.
An applicant, unaware of this instruction, uses an LLM to optimize their resume,
inadvertently triggering the AI detection.
Scenario #4: Intentional Model Influence
An attacker modifies a document in a repository used by a Retrieval-Augmented Generation
(RAG) application. When a user's query returns the modified content, the malicious...
page 12:

## Step 3: LLM rewrites the query
Ask the LLM to turn the vague question into a precise, retrieval-optimized one — same idea as the lecture's raw-query → rewritten-query table. Grounding the rewrite in the document's own category names keeps a small local model from hallucinating unrelated jargon.

In [5]:
rewrite_prompt = """Rewrite the user's question into a single, precise search query using the exact
terminology an LLM-security document would use.

Pick the closest matching category from this list (short reminder of what each covers):
- Prompt Injection: crafted inputs that hijack model instructions
- Sensitive Information Disclosure: leaking private/confidential data in outputs
- Supply Chain: risks from third-party models, data, or plugins
- Data and Model Poisoning: tampering with training data or the model itself
- Improper Output Handling: unsafe downstream use of model output (e.g. injection into other systems)
- Excessive Agency: model given too much autonomy/permissions to act
- System Prompt Leakage: system prompt contents being exposed
- Vector and Embedding Weaknesses: flaws in retrieval/embedding pipelines (e.g. RAG data leakage)
- Misinformation: hallucinated or false content presented as fact
- Unbounded Consumption: excessive resource/compute usage causing slowdowns, cost spikes, or denial of service

Reply using EXACTLY this template, filled in on one line (never output only the category name by itself):
"<Category name> - <one sentence restating the user's specific symptom>"

Example:
User question: My AI keeps making up fake statistics that sound believable
Rewritten query: Misinformation - the model fabricates plausible-sounding false statistics presented as fact

User question: {query}
Rewritten query:"""

rewritten_query = llm.invoke(rewrite_prompt.format(query=query)).content.strip()
print(f"Rewritten: {rewritten_query}")

Rewritten: Sensitive Information Disclosure - the model exposes sensitive customer information, including emails and account numbers, in its output.


## Step 4: Retrieve again with the rewritten query

In [6]:
print("After rewriting:")
for doc in vector_store.similarity_search(rewritten_query, k=3):
    print(f"page {doc.metadata['page']}: {doc.page_content}...")

After rewriting:


page 10: 2. Proprietary Algorithm Exposure
Poorly configured model outputs can reveal proprietary algorithms or data. Revealing training
data can expose models to inversion attacks, where attackers extract sensitive information
or reconstruct inputs. For instance, as demonstrated in the 'Proof Pudding' attack (CVE-2019-
20634), disclosed training data facilitated model extraction and inversion, allowing attackers
to circumvent security controls in machine learning algorithms and bypass email filters.
3. Sensitive Business Data Disclosure
Generated responses might inadvertently include confidential business information....
page 10: OWASP Top 10 for LLM Applications v2.0
7genai.owasp.org
LLM02:2025 Sensitive Information Disclosure
Description
Sensitive information can affect both the LLM and its application context. This includes personal
identifiable information (PII), financial details, health records, confidential business data, security
credentials, and legal documents. Proprietary m

## Step 5: Generate the final answer
Retrieve with the rewritten query, but still answer the user's original question.

In [7]:
retrieved_docs = vector_store.similarity_search(rewritten_query, k=3)
context = "\n\n".join(doc.page_content for doc in retrieved_docs)

prompt = f"""Answer the question based only on the following context:

{context}

Question: {query}
Answer:"""

print(llm.invoke(prompt).content)

The issue is likely due to a poorly configured model, which may be exposing proprietary algorithms or training data that contains sensitive information such as private customer data (e.g. emails and account numbers). This could be happening because the model's output is inadvertently including confidential business information, potentially leading to unauthorized access, privacy violations, and intellectual property breaches.


## Try it yourself
1. Try an even vaguer query, e.g. `"the leaking secrets thing"`, and compare rewrites.
2. Change the rewrite prompt to also expand abbreviations (e.g. "PI" → "Prompt Injection").
3. Rewrite a query that spans two categories and see which one the LLM picks.